# Análise SQL — Churn de Clientes CS

Este notebook carrega os dados processados em um banco SQLite em memória e responde perguntas de negócio com foco em Customer Success.

In [ ]:
import sqlite3
import pandas as pd

df = pd.read_csv('dados/churn_processado.csv')

conn = sqlite3.connect(':memory:')
df.to_sql('clientes', conn, index=False, if_exists='replace')

def query(sql):
    return pd.read_sql_query(sql, conn)

print(f'Dados carregados: {len(df):,} clientes')

## Query 1 — Qual é a taxa de churn geral da base?

In [ ]:
query("""
SELECT
    SUM(churn_num) AS total_churned,
    COUNT(*) AS total_clientes,
    ROUND(100.0 * SUM(churn_num) / COUNT(*), 1) AS taxa_churn
FROM clientes
""")

**Interpretação CS:** Aproximadamente 1 em cada 4 clientes cancela o serviço. Uma taxa acima de 20% é crítica para a saúde da receita recorrente e exige atenção imediata do time de Customer Success para identificar os gatilhos de cancelamento.

## Query 2 — Como o churn varia por tipo de contrato?

In [ ]:
query("""
SELECT
    Contract,
    COUNT(*) AS total,
    SUM(churn_num) AS cancelaram,
    ROUND(100.0 * SUM(churn_num) / COUNT(*), 1) AS taxa_churn
FROM clientes
GROUP BY Contract
ORDER BY taxa_churn DESC
""")

**Interpretação CS:** Clientes mensais (Month-to-month) cancelam em proporção muito maior do que clientes com contratos anuais ou bianuais. Isso indica que migrar clientes mensais para planos de maior comprometimento é uma alavanca direta de retenção — o time de CS pode atuar com campanhas de upgrade e benefícios exclusivos para contratos de longo prazo.

## Query 3 — Qual o perfil de tempo de casa e cobrança entre quem cancela e quem fica?

In [ ]:
query("""
SELECT
    CASE WHEN churn_num = 1 THEN 'Cancelou' ELSE 'Ativo' END AS situacao,
    ROUND(AVG(tenure), 1) AS tempo_medio_meses,
    ROUND(AVG(MonthlyCharges), 2) AS cobranca_media
FROM clientes
GROUP BY situacao
""")

**Interpretação CS:** Clientes que cancelam têm tempo de casa significativamente menor e cobranças mensais mais altas, o que sugere que novos clientes com planos premium são os mais vulneráveis. O onboarding nos primeiros 12 meses é a janela crítica de intervenção para o CS.

## Query 4 — Quantos clientes ativos estão em situação de alto risco de churn?

In [ ]:
query("""
SELECT COUNT(*) AS clientes_alto_risco
FROM clientes
WHERE Contract = 'Month-to-month'
  AND tenure <= 12
  AND MonthlyCharges > (SELECT AVG(MonthlyCharges) FROM clientes)
  AND churn_num = 0
""")

**Interpretação CS:** Esses clientes reúnem todos os fatores de risco: contrato mensal, pouco tempo de casa e cobrança acima da média. Eles ainda estão ativos e representam uma oportunidade concreta de ação preventiva — contato proativo, oferta de upgrade de plano ou programa de fidelidade.

## Query 5 — Qual a receita em risco por tipo de contrato?

In [ ]:
query("""
SELECT
    Contract,
    SUM(CASE WHEN churn_num = 1 THEN MonthlyCharges ELSE 0 END) AS receita_perdida,
    ROUND(100.0 * SUM(CASE WHEN churn_num = 1 THEN MonthlyCharges ELSE 0 END)
          / SUM(MonthlyCharges), 1) AS pct_receita_em_risco
FROM clientes
GROUP BY Contract
ORDER BY receita_perdida DESC
""")

**Interpretação CS:** A maior parte da receita perdida para churn vem de clientes mensais, reforçando a prioridade de retenção nesse segmento. Contratos anuais e bianuais, apesar de menores em volume de churn, quando cancelam geram perda pontual relevante — justificando atenção no momento de renovação.

## Query 6 — O TechSupport reduz o churn?

In [ ]:
query("""
SELECT
    TechSupport,
    COUNT(*) AS total,
    ROUND(100.0 * SUM(churn_num) / COUNT(*), 1) AS taxa_churn
FROM clientes
WHERE TechSupport != 'Não contratado'
GROUP BY TechSupport
ORDER BY taxa_churn DESC
""")

**Interpretação CS:** Clientes com TechSupport ativo apresentam taxa de churn consideravelmente menor. Isso sugere que o suporte técnico age como âncora de relacionamento — clientes que se sentem amparados tendem a permanecer. Uma alavanca de CS seria incentivar a adoção do TechSupport durante o onboarding.

## Query 7 — Como o churn varia conforme o tempo de casa do cliente?

In [ ]:
query("""
SELECT
    tenure_grupo,
    COUNT(*) AS total,
    SUM(churn_num) AS cancelaram,
    ROUND(100.0 * SUM(churn_num) / COUNT(*), 1) AS taxa_churn
FROM clientes
GROUP BY tenure_grupo
ORDER BY taxa_churn DESC
""")

**Interpretação CS:** O primeiro ano é o período de maior risco de churn. Clientes com mais de 4 anos de casa cancelam em proporção muito menor, indicando que fidelidade cresce com o tempo. O time de CS deve concentrar esforços de onboarding e sucesso inicial nos primeiros 12 meses para aumentar as chances de retenção de longo prazo.

## Query 8 — Qual é o perfil completo de quem cancela vs quem permanece?

In [ ]:
query("""
SELECT
    CASE WHEN churn_num = 1 THEN 'Cancelou' ELSE 'Ativo' END AS situacao,
    ROUND(AVG(tenure), 1) AS media_meses_de_casa,
    ROUND(AVG(MonthlyCharges), 2) AS media_cobranca_mensal,
    ROUND(AVG(TotalCharges), 2) AS media_total_gasto
FROM clientes
GROUP BY situacao
""")

**Interpretação CS:** O cliente que cancela paga mais por mês, tem menos tempo de casa e acumula muito menos gasto total — ou seja, sai antes de gerar valor de longo prazo para o negócio. Esse perfil é o da oportunidade perdida: o cliente chegou disposto a pagar, mas não encontrou razões suficientes para ficar. Reduzir esse gap é a missão central do Customer Success.

In [ ]:
conn.close()
print('Conexão encerrada.')